# PEFT+AFSP — scoring the stacked rungs

---
## 1 — Preconditions

In [ ]:
%cd /home/prnamhr/projects/Style-Aware-MT
import json
import os
import subprocess
import sys
from pathlib import Path

import yaml

PY = sys.executable
COMET_PY = '.venv-comet/bin/python'
SPLIT = 'val'
OUT = Path('outputs')

REFERENCE, CONTROL, ARM = 'peft', 'peft_knn', 'peft_afsp'
CONDS = [REFERENCE, CONTROL, ARM]

METRICS = ('chrf', 'bleu', 'comet')

N_BOOT, N_STYLO, ALPHA, SEED = 10000, 2000, 0.05, 42
print(f'{len(CONDS)} conditions, {len(METRICS)} adequacy metrics, {N_BOOT} resamples, seed {SEED}')

/home/prnamhr/projects/Style-Aware-MT
3 conditions, 3 adequacy metrics, 10000 resamples, seed 42


In [28]:
from pathlib import Path

if not Path('manage.py').exists():
    if not Path('Style-Aware-MT/manage.py').exists():
        !git clone https://github.com/prnamhr/Style-Aware-MT.git
    %cd Style-Aware-MT
!git pull --ff-only
!git rev-parse --short HEAD

Already up to date.
72a3c1a


In [29]:
VAL = [json.loads(x) for x in Path(f'data/splits/{SPLIT}.jsonl').open(encoding='utf-8') if x.strip()]
SRC = [r['input'] for r in VAL]

ROWS = {}
for cond in CONDS:
    path = OUT / f'{cond}_{SPLIT}.jsonl'
    assert path.exists(), f'{path} missing -- sync it back from the generation box'
    ROWS[cond] = [json.loads(x) for x in path.open(encoding='utf-8') if x.strip()]
    assert len(ROWS[cond]) == len(VAL), f'{cond}: {len(ROWS[cond])} rows, expected {len(VAL)}'
    assert [r['input'] for r in ROWS[cond]] == SRC, f'{cond}: not paired with {SPLIT}.jsonl'
    blank = sum(1 for r in ROWS[cond] if not r['prediction'].strip())
    assert blank == 0, f'{cond}: {blank} blank predictions break the feature matrix pairing'
    print(f'{cond:10s} {len(ROWS[cond])} rows, aligned, 0 blank')

identical = sum(1 for a, b in zip(ROWS[CONTROL], ROWS[ARM]) if a['prediction'] == b['prediction'])
print(f'\n{identical / len(VAL):.1%} of segments are decoded identically by {CONTROL} and {ARM}: '
      f'the two differ only in which exemplars the prompt carried.')

peft       1323 rows, aligned, 0 blank
peft_knn   1323 rows, aligned, 0 blank
peft_afsp  1323 rows, aligned, 0 blank

11.3% of segments are decoded identically by peft_knn and peft_afsp: the two differ only in which exemplars the prompt carried.


In [30]:
MANIFEST = json.loads((OUT / 'peft_afsp_manifest.json').read_text(encoding='utf-8'))
PEFT_CFG = yaml.safe_load(Path('configs/peft_qwen.yaml').read_text(encoding='utf-8'))
GEN, PEFT_GEN = MANIFEST['generator'], PEFT_CFG['generator']

for key in ('model', 'adapter_path', 'temperature', 'top_p', 'seed', 'max_tokens',
            'dtype', 'load_in_4bit'):
    assert GEN[key] == PEFT_GEN[key], f'{key}: {GEN[key]} vs peft {PEFT_GEN[key]}'
assert set(MANIFEST['conditions']) == {CONTROL, ARM}, MANIFEST['conditions']

idx = MANIFEST['index']
print('decoding matches peft on all eight settings')
print(f"adapter {MANIFEST['adapter']['sha256'][:12]}  r={MANIFEST['adapter']['r']}")
print(f"index rebuilt_here={idx['rebuilt_here']}  reproduced={idx.get('reproduced')}")
print(f"device {MANIFEST['versions']['device']}, torch {MANIFEST['versions']['torch']}")
if idx.get('reproduced') is False:
    print('\nWARNING: the index differs from the one afsp_full used. The contrasts below are'
          '\nunaffected -- all three conditions come from one box -- but any comparison to the'
          '\nJuly afsp_full row must say so.')

decoding matches peft on all eight settings
adapter ad97c46af852  r=32
index rebuilt_here=False  reproduced=None
device NVIDIA GeForce RTX 4090, torch 2.12.0+cu130


---
## 2 — The detection floor

In [ ]:
DECOMP = json.loads(Path(f'results/heldout_decomp_{SPLIT}.json').read_text(encoding='utf-8'))
PROMPTING = json.loads(
    Path(f'results/heldout_decomp_prompting_{SPLIT}.json').read_text(encoding='utf-8'))

for report in (DECOMP, PROMPTING):
    assert report['reference'] == REFERENCE, report['reference']
    boot = report['bootstrap']
    assert (boot['n_resamples'], boot['seed'], boot['alpha']) == (N_BOOT, SEED, ALPHA), boot
    assert boot['n_segments'] == len(VAL), boot

BASE = PROMPTING['cells'][REFERENCE]['dist_heldout']
CEILING = PROMPTING['cells']['afsp_full']['dist_heldout']
HALF = {c: (v['dist_heldout_delta']['ci_high'] - v['dist_heldout_delta']['ci_low']) / 2
        for c, v in DECOMP['cells'].items() if 'dist_heldout_delta' in v}
FLOOR = max(HALF.values())

for cond, h in HALF.items():
    print(f'{cond:14s} half-width against {REFERENCE} {h:.4f}')
print(f'\nfloor {FLOOR:.4f}: a shift of that size or below is not distinguishable from resampling '
      f'noise at n={len(VAL)}.')
print(f'P1 predicts {ARM} lands between peft {BASE:.4f} and afsp_full {CEILING:.4f}.')

rlsf_w3_0.0    half-width against peft 0.0186
rlsf_w3_2.0    half-width against peft 0.0245
rlsf_w3_6.0    half-width against peft 0.0204

floor 0.0245: a shift of that size or below is not distinguishable from resampling noise at n=1323.
P1 predicts peft_afsp lands between peft 0.1707 and afsp_full 0.2990.


---
## 3 — Adequacy: chrF, BLEU, COMET

In [32]:
!{PY} manage.py eval --conditions {' '.join(CONDS)} --split {SPLIT}

condition  n     BLEU   chrF   marker_rate  ref_marker_rate
-----------------------------------------------------------
peft       1323  16.9   41.58  0.92         0.79           
peft_knn   1323  17.98  42.4   0.78         0.79           
peft_afsp  1323  17.77  42.12  0.79         0.79           


In [ ]:
from src.eval.quick import score

SURFACE = {c: score(c, OUT, SPLIT) for c in CONDS}
print(f"{'condition':12s} {'chrF':>8s} {'BLEU':>8s} {'markers/seg':>12s}")
for cond in CONDS:
    s = SURFACE[cond]
    print(f"{cond:12s} {s['chrF']:8.2f} {s['BLEU']:8.2f} {s['marker_rate']:12.2f}")
print(f"gold targets carry {SURFACE[REFERENCE]['ref_marker_rate']:.2f} markers per segment")

condition        chrF     BLEU  markers/seg
peft            41.58    16.90         0.92
peft_knn        42.40    17.98         0.78
peft_afsp       42.12    17.77         0.79
gold targets carry 0.79 markers per segment


In [34]:
if not Path(COMET_PY).exists():
    pip = [COMET_PY, '-m', 'pip', 'install', '-q']
    subprocess.run([PY, '-m', 'venv', '.venv-comet'], check=True)
    subprocess.run([*pip, '--upgrade', 'pip'], check=True)
    subprocess.run([*pip, 'setuptools<81'], check=True)
    subprocess.run([*pip, '-r', 'requirements-comet.txt'], check=True)

COMET_PATH = f'results/comet_{SPLIT}.json'
PRIOR = set(json.loads(Path(COMET_PATH).read_text(encoding='utf-8')))
for cond in (CONTROL, ARM):
    subprocess.run(
        [COMET_PY, 'manage.py', 'comet', '--conditions', cond, '--split', SPLIT,
         '--results_path', COMET_PATH, '--batch_size', '16'],
        check=True,
    )

Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 3374.88it/s]
Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
Encoder model frozen.
/home/prnamhr/projects/Style-Aware-MT/.venv-comet/lib/python3.11/site-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and ver

peft_knn         COMET 0.7015  (n=1323)
preserved 10 condition(s) not scored here: afsp_full, afsp_margin, commercial_haiku, knn_fewshot, peft, random_fewshot, rlsf_w3_0.0, rlsf_w3_2.0, rlsf_w3_6.0, zeroshot
Wrote results/comet_val.json


Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 1984.06it/s]
Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
Encoder model frozen.
/home/prnamhr/projects/Style-Aware-MT/.venv-comet/lib/python3.11/site-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and ver

peft_afsp        COMET 0.7033  (n=1323)
preserved 11 condition(s) not scored here: afsp_full, afsp_margin, commercial_haiku, knn_fewshot, peft, peft_knn, random_fewshot, rlsf_w3_0.0, rlsf_w3_2.0, rlsf_w3_6.0, zeroshot
Wrote results/comet_val.json


In [35]:
COMET = json.loads(Path(COMET_PATH).read_text(encoding='utf-8'))

assert PRIOR <= set(COMET), f'lost from {COMET_PATH}: {sorted(PRIOR - set(COMET))}'
ref = COMET[REFERENCE]
for cond in (CONTROL, ARM):
    rec = COMET[cond]
    assert rec['n'] == len(VAL), (cond, rec['n'])
    assert rec['model'] == ref['model'], (cond, rec['model'])
    assert rec['sources'] == ref['sources'], f'{cond}: segments are not paired with {REFERENCE}'

print(f"{ref['model']} scored all three on the same {len(VAL)} segments; "
      f'{len(PRIOR)} conditions already in the file preserved')
for cond in CONDS:
    print(f"{cond:12s} COMET {COMET[cond]['system']:.4f}")

Unbabel/wmt22-comet-da scored all three on the same 1323 segments; 10 conditions already in the file preserved
peft         COMET 0.6986
peft_knn     COMET 0.7015
peft_afsp    COMET 0.7033


In [36]:
# One paired bootstrap per metric, each writing its own table. --adjacent adds the
# peft_afsp - peft_knn pair that P3 is scored on; the other two are against peft.
for metric in METRICS:
    r = subprocess.run([PY, 'manage.py', 'bootstrap', '--metric', metric, '--split', SPLIT,
                        '--adjacent', '--conditions', *CONDS, '--baseline', REFERENCE,
                        '--n_resamples', str(N_BOOT), '--alpha', str(ALPHA), '--seed', str(SEED),
                        '--out', f'results/bootstrap_{metric}_peft_afsp_{SPLIT}.json'],
                       check=False)
    assert r.returncode == 0, f'{metric} bootstrap exited {r.returncode}'

wrote results/bootstrap_chrf_peft_afsp_val.json

chrf paired bootstrap  (resamples=10000, split=val)
comparison            n     diff    ci95             p       sig
----------------------------------------------------------------
peft_knn - peft       1323  0.964   [0.403, 1.527]   0.001   *  
peft_afsp - peft      1323  0.901   [0.330, 1.473]   0.0014  *  
peft_afsp - peft_knn  1323  -0.063  [-0.575, 0.465]  0.831      

* = 95% CI excludes 0 (difference significant at α=0.05)
wrote results/bootstrap_bleu_peft_afsp_val.json

bleu paired bootstrap  (resamples=10000, split=val)
comparison            n     diff    ci95             p       sig
----------------------------------------------------------------
peft_knn - peft       1323  0.825   [0.262, 1.384]   0.0048  *  
peft_afsp - peft      1323  0.759   [0.201, 1.308]   0.0062  *  
peft_afsp - peft_knn  1323  -0.066  [-0.603, 0.476]  0.8248     

* = 95% CI excludes 0 (difference significant at α=0.05)
wrote results/bootstrap_comet_pe

---
## 4 — Register fit

In [37]:
!{PY} manage.py stylometrics --conditions {' '.join(CONDS)} --split {SPLIT} --targets-split train

label         n      lex_density  lex_density_sd  ttr     ttr_sd  root_ttr  root_ttr_sd  sent_len_mean  sent_len_mean_sd  sent_len_var  sent_len_var_sd  marker_rate  marker_rate_sd  stylo_dist
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
target:train  10860  0.4344       0.1101          0.854   0.1085  4.0437    1.0426       24.7388        16.6125           6.6894        58.4011          0.0327       0.0567          0.0       
peft          1323   0.4088       0.1118          0.8515  0.1229  3.9389    1.0278       25.4086        29.1189           3.6097        38.2494          0.0404       0.0621          0.2886    
peft_knn      1323   0.4108       0.1071          0.8366  0.1242  3.8745    0.9858       24.8611        17.0783           3.148         42.3741          0.0347       0.0588          0.3152    
peft_afsp     1323   0.4134       0

In [38]:
!{PY} manage.py stylometrics_ci --split {SPLIT} --conditions {' '.join(CONDS)} \
    --n_resamples {N_STYLO} --alpha {ALPHA} --seed {SEED} \
    --results_path results/stylometrics_ci_peft_afsp_{SPLIT}.json


Register fit of the main conditions  (split=val, n=1323 segments, resamples=2000, seed=42)
stylo_dist = standardized distance to the target-register centroid; lower is better.

rank  condition  stylo_dist  ci95              P(this rank)  modal rank  mean rank
----------------------------------------------------------------------------------
1     peft_afsp  0.2701      [0.2292, 0.3217]  0.796         1 (0.796)   1.20     
2     peft       0.2886      [0.2410, 0.3466]  0.662         2 (0.662)   1.93     
3     peft_knn   0.3152      [0.2730, 0.3666]  0.864         3 (0.864)   2.86     

Signed z per register feature (95% CI; 0 = on target)
condition  lex_density              ttr                      root_ttr                 marker_rate           
------------------------------------------------------------------------------------------------------------
peft_afsp  -0.191 [-0.242, -0.136]  -0.109 [-0.171, -0.051]  -0.148 [-0.198, -0.098]  +0.051 [-0.004, 0.107]
peft       -0.233 [-0.289

In [ ]:
new = json.loads(Path(f'results/stylometrics_ci_peft_afsp_{SPLIT}.json').read_text(encoding='utf-8'))
old = json.loads(Path(f'results/stylometrics_ci_{SPLIT}.json').read_text(encoding='utf-8'))
a, b = new['cells'][REFERENCE], old['cells'][REFERENCE]
assert a['stylo_dist'] == b['stylo_dist'] and a['z'] == b['z'], 'peft moved between passes'
print(f"peft reproduces the committed row: stylo_dist {a['stylo_dist']:.4f}")

peft reproduces the committed row: stylo_dist 0.2886


---
## 5 — The declared axis: held-out register distance

In [40]:
!{PY} manage.py heldout_decomp --split {SPLIT} --no-figure \
    --conditions {' '.join(CONDS)} --reference {REFERENCE} \
    --n_resamples {N_BOOT} --alpha {ALPHA} --seed {SEED} \
    --results_path results/heldout_decomp_peft_afsp_{SPLIT}.json


Held-out register distance, decomposed  (split=val, n=1323 segments, resamples=10000, seed=42)
distance over ttr, root_ttr, marker_rate; delta and share are against peft. Lower distance is better.

condition  w3  feature      z        z ci95            d vs peft  z^2      share   dist  
-----------------------------------------------------------------------------------------
peft       0   ttr          -0.0227  [-0.084, +0.038]  +0.0000    0.00052    1.8%  0.1707
peft       0   root_ttr     -0.1005  [-0.154, -0.046]  +0.0000    0.01010   34.7%  0.1707
peft       0   marker_rate  +0.1361  [+0.080, +0.195]  +0.0000    0.01852   63.6%  0.1707
peft_knn   0   ttr          -0.1599  [-0.222, -0.098]  -0.1373*   0.02557   48.1%  0.2306
peft_knn   0   root_ttr     -0.1623  [-0.214, -0.111]  -0.0619*   0.02635   49.5%  0.2306
peft_knn   0   marker_rate  +0.0356  [-0.018, +0.090]  -0.1005*   0.00126    2.4%  0.2306
peft_afsp  0   ttr          -0.1094  [-0.170, -0.049]  -0.0869*   0.01197   32.9%

In [41]:
!{PY} manage.py heldout_decomp --split {SPLIT} --no-figure \
    --conditions {CONTROL} {ARM} --reference {CONTROL} \
    --n_resamples {N_BOOT} --alpha {ALPHA} --seed {SEED} \
    --results_path results/heldout_decomp_peft_afsp_vs_knn_{SPLIT}.json


Held-out register distance, decomposed  (split=val, n=1323 segments, resamples=10000, seed=42)
distance over ttr, root_ttr, marker_rate; delta and share are against peft_knn. Lower distance is better.

condition  w3  feature      z        z ci95            d vs peft_knn  z^2      share   dist  
---------------------------------------------------------------------------------------------
peft_knn   0   ttr          -0.1599  [-0.222, -0.098]  +0.0000        0.02557   48.1%  0.2306
peft_knn   0   root_ttr     -0.1623  [-0.214, -0.111]  +0.0000        0.02635   49.5%  0.2306
peft_knn   0   marker_rate  +0.0356  [-0.018, +0.090]  +0.0000        0.00126    2.4%  0.2306
peft_afsp  0   ttr          -0.1094  [-0.170, -0.049]  +0.0504*       0.01197   32.9%  0.1908
peft_afsp  0   root_ttr     -0.1477  [-0.199, -0.097]  +0.0144        0.02182   60.0%  0.1908
peft_afsp  0   marker_rate  +0.0510  [-0.005, +0.109]  +0.0156        0.00261    7.2%  0.1908

Reward-side features, same decomposition aga

In [ ]:
D_REF = json.loads(Path(f'results/heldout_decomp_peft_afsp_{SPLIT}.json').read_text(encoding='utf-8'))
D_CTL = json.loads(
    Path(f'results/heldout_decomp_peft_afsp_vs_knn_{SPLIT}.json').read_text(encoding='utf-8'))
assert D_REF['cells'][REFERENCE]['dist_heldout'] == DECOMP['cells'][REFERENCE]['dist_heldout']
assert D_REF['adjacent_in_omega'] == [], 'these rungs share a judge weight; no omega contrast'
print(f"peft dist_heldout {D_REF['cells'][REFERENCE]['dist_heldout']:.4f}, matches the committed file")

peft dist_heldout 0.1707, matches the committed file


---
## 6 — Read-out

In [43]:
def holm(tests, alpha=ALPHA):
    """Holm-Bonferroni over one claim's family; returns name -> (p, survives)."""
    ordered = sorted(tests.items(), key=lambda kv: kv[1])
    out, blocked = {}, False
    for i, (name, p) in enumerate(ordered):
        ok = p <= alpha / (len(ordered) - i)
        blocked = blocked or not ok
        out[name] = (p, not blocked)
    return out


def line(label, delta, lo, hi, p, sig, places=4):
    mark = '*' if sig else ' '
    return f'  {label:26s} {delta:+.{places}f} [{lo:+.{places}f}, {hi:+.{places}f}]  p={p:.4f} {mark}'


PLACES = {'chrf': 2, 'bleu': 2, 'comet': 4}
BOOT = {m: json.loads(
    Path(f'results/bootstrap_{m}_peft_afsp_{SPLIT}.json').read_text(encoding='utf-8'))
        for m in METRICS}
STYLO = json.loads(Path(f'results/stylometrics_ci_peft_afsp_{SPLIT}.json').read_text(encoding='utf-8'))


def comparison(metric, a, b):
    for rec in BOOT[metric]['comparisons']:
        if (rec['a'], rec['b']) == (a, b):
            return rec
    raise KeyError(f'{metric}: {a} - {b} not in the table')

In [44]:
# P1 -- held-out register distance worsens against peft, by the marker_rate mechanism.
d = D_REF['cells'][ARM]['dist_heldout_delta']
arm = D_REF['cells'][ARM]['dist_heldout']

print('P1  dist_heldout, peft_afsp against peft')
print(f'  peft {BASE:.4f} -> peft_afsp {arm:.4f}   (afsp_full on the frozen base: {CEILING:.4f})')
print(line('peft_afsp - peft', d['delta'], d['ci_low'], d['ci_high'], d['p_value'], d['significant']))
c = D_REF['cells'][CONTROL]['dist_heldout_delta']
print(line('peft_knn - peft', c['delta'], c['ci_low'], c['ci_high'], c['p_value'], c['significant']))

worsens = d['significant'] and d['delta'] > 0
between = BASE < arm < CEILING
print(f'\n  predicted: rises above {BASE:.4f}, below {CEILING:.4f}, interval clear of zero')
print(f"  observed move {d['delta']:+.4f} against a {FLOOR:.4f} floor")
print(f"  P1 {'HOLDS' if (worsens and between) else 'FAILS'}"
      f"{'' if d['significant'] else '  (interval straddles zero -- no movement detected)'}")

# The three held-out features, read beside where afsp_full sits on each: P1 names the
# direction, not just the size, and marker_rate is the one it names.
print(f"\n  {'feature':12s} {'peft':>8s} {'peft_afsp':>10s} {'afsp_full':>10s} {'share':>7s}")
for name in ('ttr', 'root_ttr', 'marker_rate'):
    f = D_REF['cells'][ARM]['features'][name]
    r = D_REF['cells'][REFERENCE]['features'][name]
    t = PROMPTING['cells']['afsp_full']['features'][name]
    print(f"  {name:12s} {r['z']:+8.4f} {f['z']:+10.4f} {t['z']:+10.4f} {f['share']:7.1%}")

P1  dist_heldout, peft_afsp against peft
  peft 0.1707 -> peft_afsp 0.1908   (afsp_full on the frozen base: 0.2990)
  peft_afsp - peft           +0.0202 [-0.0289, +0.0713]  p=0.4344  
  peft_knn - peft            +0.0589 [-0.0004, +0.1170]  p=0.0512  

  predicted: rises above 0.1707, below 0.2990, interval clear of zero
  observed move +0.0202 against a 0.0245 floor
  P1 FAILS  (interval straddles zero -- no movement detected)

  feature          peft  peft_afsp  afsp_full   share
  ttr           -0.0227    -0.1094    -0.1090   32.9%
  root_ttr      -0.1005    -0.1477    -0.1779   60.0%
  marker_rate   +0.1361    +0.0510    +0.2142    7.2%


In [49]:
h = D_CTL['cells'][ARM]['dist_heldout_delta']
tests = {'dist_heldout': h['p_value']}
print(f'P3  {ARM} against {CONTROL}')
print(line('dist_heldout', h['delta'], h['ci_low'], h['ci_high'], h['p_value'], h['significant']))
for m in METRICS:
    rec = comparison(m, ARM, CONTROL)
    tests[m] = rec['p_value']
    print(line(m, rec['diff'], rec['ci_low'], rec['ci_high'], rec['p_value'], rec['significant'],
               PLACES[m]))
for rec in STYLO['paired_all']:
    if {rec['a'], rec['b']} != {ARM, CONTROL}:
        continue
    flip = -1.0 if rec['a'] == CONTROL else 1.0
    lo, hi = sorted((flip * rec['ci_low'], flip * rec['ci_high']))
    tests['stylo_dist'] = rec['p_value']
    print(line('stylo_dist', flip * rec['diff'], lo, hi, rec['p_value'], rec['significant']))
assert len(tests) == 5, f'expected five quantities, got {sorted(tests)}'


P3  peft_afsp against peft_knn
  dist_heldout               -0.0387 [-0.0714, -0.0067]  p=0.0190 *
  chrf                       -0.06 [-0.58, +0.46]  p=0.8310  
  bleu                       -0.07 [-0.60, +0.48]  p=0.8248  
  comet                      +0.0017 [-0.0013, +0.0047]  p=0.2614  
  stylo_dist                 -0.0442 [-0.0771, -0.0127]  p=0.0040 *


In [ ]:
print('P4  adequacy against peft')
p4 = {}
for m in METRICS:
    for cond in (CONTROL, ARM):
        rec = comparison(m, cond, REFERENCE)
        p4[f'{m}:{cond}'] = rec
        print(line(f'{cond} - peft, {m}', rec['diff'], rec['ci_low'], rec['ci_high'],
                   rec['p_value'], rec['significant'], PLACES[m]))

declared = {k: r for k, r in p4.items() if not k.startswith('comet')}
rose = [k for k, r in declared.items() if r['significant'] and r['diff'] > 0]
print('\n  predicted: chrF and BLEU hold or fall, no significant rise')
print(f"  P4 {'FAILS -- ' + ', '.join(rose) + ' rose' if rose else 'HOLDS'}")

# Corpus against segment mean, the two aggregates of section 3. They can disagree in sign on
# a small gap; the selection rule ranks on the corpus column.
print(f"\n  {'pair':22s} {'corpus':>9s} {'segment mean':>14s}")
for m in ('chrf', 'bleu'):
    key = {'chrf': 'chrF', 'bleu': 'BLEU'}[m]
    for cond in (CONTROL, ARM):
        rec = comparison(m, cond, REFERENCE)
        corpus = SURFACE[cond][key] - SURFACE[REFERENCE][key]
        print(f"  {cond + ' - peft, ' + m:22s} {corpus:+9.2f} {rec['diff']:+14.2f}")

P4  adequacy against peft
  peft_knn - peft, chrf      +0.96 [+0.40, +1.53]  p=0.0010 *
  peft_afsp - peft, chrf     +0.90 [+0.33, +1.47]  p=0.0014 *
  peft_knn - peft, bleu      +0.82 [+0.26, +1.38]  p=0.0048 *
  peft_afsp - peft, bleu     +0.76 [+0.20, +1.31]  p=0.0062 *
  peft_knn - peft, comet     +0.0029 [-0.0008, +0.0066]  p=0.1208  
  peft_afsp - peft, comet    +0.0047 [+0.0011, +0.0081]  p=0.0104 *

  predicted: chrF and BLEU hold or fall, no significant rise
  P4 FAILS -- chrf:peft_knn, chrf:peft_afsp, bleu:peft_knn, bleu:peft_afsp rose

  pair                      corpus   segment mean
  peft_knn - peft, chrf      +0.82          +0.96
  peft_afsp - peft, chrf     +0.54          +0.90
  peft_knn - peft, bleu      +1.08          +0.82
  peft_afsp - peft, bleu     +0.87          +0.76


In [ ]:
if not d['significant']:
    print('No quadrant. dist_heldout does not separate from peft, and if nothing else above')
    print("separates either, the entry's uninformative-outcome clause applies: this is a null")
    print('result of stacking, not evidence that the two families compose.')
else:
    row = 'Q3 / Q4' if d['delta'] > 0 else 'Q1 / Q2'
    axis = 'worsens' if d['delta'] > 0 else 'improves'
    print(f"Distance {axis}: the pass sits in {row}, split by a judge reading not taken.")
    if d['delta'] > 0:
        moved = D_REF['cells'][CONTROL]['dist_heldout_delta']['significant']
        cause = 'Exemplars at all -- prompt-format mismatch.' if moved else 'AFSP selection.'
        print(f'  peft_knn also moves: {moved}. {cause}')
        print("  Q3 (Phi rises) would put the ladder's decoupling outside GRPO and is the one")
        print('  result here worth ~$1.75 of rater to resolve. Q4 (Phi flat or down) is the')
        print('  predicted outcome and needs no purchase.')
print('\nP2 is UNSCORED. No judge call was made in this pass, by design.')

No quadrant. dist_heldout does not separate from peft, and if nothing else above
separates either, the entry's uninformative-outcome clause applies: this is a null
result of stacking, not evidence that the two families compose.

P2 is UNSCORED. No judge call was made in this pass, by design.
